# Graph Analytics — Industri × Region Clustering
### Capstone Project: Screening Credit Agentic AI for eLO System

Notebook ini menambahkan modul **graph analytics** ke pipeline capstone, dengan use case:
**mendeteksi konsentrasi risiko dan inkonsistensi scoring berdasarkan kombinasi `sub_industri` × `region`**,
menggunakan graph nasabah-ke-nasabah yang diproyeksikan dari kesamaan atribut.

**Alur (Tahap 0–7):**
0. Load & validasi data
1. Bangun bipartite graph (Nasabah ↔ Atribut komposit sub_industri|region)
2. Proyeksikan ke graph Nasabah–Nasabah (dengan bobot tambahan)
3. Community detection (Louvain)
4. Hitung metrik risiko per cluster
5. Centrality — cari nasabah representatif per cluster
6. Visualisasi graph di level cluster
7. Generate insight/narasi otomatis

> Catatan: sesuaikan `DATA_PATH` dan `COLUMN MAPPING` di Tahap 0 dengan nama file & kolom asli di `master_dataset.csv` / `master_scored.csv` kamu.

## Tahap 0 — Setup & Load Data

Kolom yang dibutuhkan minimal:
- ID nasabah (`nik`)
- `industri`, `sub_industri`
- `region` / `branch_name`
- `eligibility_score` (dan/atau `decision`, `zone`) dari hasil scoring (master_scored.csv)

Kalau nama kolom di file kamu berbeda, ubah bagian **COLUMN MAPPING** di bawah — sisanya di notebook ini
mereferensikan lewat variabel `COL`, jadi tidak perlu ganti manual di banyak tempat.

In [ ]:
import pandas as pd
import numpy as np
import itertools
import networkx as nx
from networkx.algorithms import bipartite

# --- Community detection: pakai networkx built-in (v3.0+), fallback ke python-louvain kalau ada ---
try:
    from networkx.algorithms.community import louvain_communities
    HAVE_NX_LOUVAIN = True
except ImportError:
    HAVE_NX_LOUVAIN = False

try:
    import community as community_louvain  # python-louvain, pip install python-louvain
    HAVE_PY_LOUVAIN = True
except ImportError:
    HAVE_PY_LOUVAIN = False

import matplotlib.pyplot as plt
import matplotlib.cm as cm

pd.set_option("display.max_columns", 50)
np.random.seed(42)

In [ ]:
# ====== EDIT SESUAI FILE KAMU ======
DATA_PATH = "master_scored.csv"   # atau "master_dataset.csv" kalau eligibility_score/decision belum ada di sini

# ====== COLUMN MAPPING — sesuaikan kalau nama kolom di file kamu beda ======
COL = {
    "id": "nik",
    "industri": "industri",
    "sub_industri": "sub_industri",
    "region": "region",          # fallback ke branch_name kalau region tidak ada
    "branch": "branch_name",
    "score": "eligibility_score",
    "decision": "decision",      # Layak / Layak Bersyarat / Perlu Review Ulang / Tidak Layak
}

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df[[c for c in COL.values() if c in df.columns]].head()

In [ ]:
# --- Validasi kolom yang tersedia ---
missing = [k for k, v in COL.items() if v not in df.columns]
if missing:
    print("Kolom belum ketemu di dataframe, cek/ubah COLUMN MAPPING di atas:", missing)
else:
    print("Semua kolom mapping ditemukan. Lanjut ke Tahap 1.")

# fallback region -> branch_name kalau region tidak ada
if COL["region"] not in df.columns and COL["branch"] in df.columns:
    COL["region"] = COL["branch"]
    print(f"Fallback: pakai '{COL['branch']}' sebagai region.")

df = df.dropna(subset=[COL["id"], COL["sub_industri"], COL["region"]]).copy()
df[COL["id"]] = df[COL["id"]].astype(str)
print("Jumlah nasabah valid:", df[COL["id"]].nunique())

## Tahap 1 — Bangun Bipartite Graph (Nasabah ↔ Atribut Komposit)

Setiap nasabah dihubungkan ke satu node atribut komposit `"sub_industri|region"`.
Node komposit dipilih (bukan sub_industri atau region terpisah) supaya cluster yang muncul nanti
langsung menangkap interaksi dua dimensi sekaligus, bukan cuma satu kategori tunggal.

In [ ]:
df["attr_node"] = df[COL["sub_industri"]].astype(str) + "|" + df[COL["region"]].astype(str)

B = nx.Graph()

# node nasabah (bipartite=0) dan node atribut (bipartite=1)
nasabah_nodes = df[COL["id"]].tolist()
attr_nodes = df["attr_node"].unique().tolist()

B.add_nodes_from(nasabah_nodes, bipartite=0)
B.add_nodes_from(attr_nodes, bipartite=1)

edges = list(zip(df[COL["id"]], df["attr_node"]))
B.add_edges_from(edges)

print(f"Bipartite graph: {len(nasabah_nodes)} nasabah, {len(attr_nodes)} atribut komposit, {B.number_of_edges()} edge")
print("\nContoh atribut komposit terbesar:")
print(df["attr_node"].value_counts().head(10))

## Tahap 2 — Proyeksi ke Graph Nasabah–Nasabah

Dua nasabah mendapat edge kalau mereka menempel ke atribut komposit yang sama (sub_industri & region sama).
Bobot dasar dari proyeksi bipartite = jumlah atribut bersama (di sini selalu 1, karena tiap nasabah cuma
punya 1 atribut komposit). Kita tambahkan **bobot bonus** dari dua sinyal lain supaya edge lebih informatif:

- **+bonus** kalau `branch_name` juga sama (lokasi cabang identik, bukan cuma region)
- **+bonus** kalau `eligibility_score` mirip (profil risiko berdekatan)

> Catatan skala: kalau satu grup sub_industri×region berisi ratusan nasabah, hasil proyeksinya jadi
> *hampir complete graph* di dalam grup itu — ini **wajar**, karena tujuan proyeksi ini memang
> mengelompokkan berdasarkan kombinasi atribut. Untuk visualisasi (Tahap 6) kita akan turun ke level
> cluster, bukan node individual, supaya tetap terbaca.

In [ ]:
nasabah_only = {n for n, d in B.nodes(data=True) if d["bipartite"] == 0}
G = bipartite.weighted_projected_graph(B, nasabah_only)

print(f"Graph nasabah-nasabah: {G.number_of_nodes()} node, {G.number_of_edges()} edge (sebelum bonus)")

# --- Tambah bonus weight dari branch_name sama & score mirip ---
score_map = df.set_index(COL["id"])[COL["score"]].to_dict() if COL["score"] in df.columns else {}
branch_map = df.set_index(COL["id"])[COL["branch"]].to_dict() if COL["branch"] in df.columns else {}

SCORE_SIM_THRESHOLD = 0.05   # anggap "mirip" kalau selisih eligibility_score <= 0.05

for u, v, d in G.edges(data=True):
    bonus = 0
    if branch_map and branch_map.get(u) == branch_map.get(v):
        bonus += 1
    if score_map and u in score_map and v in score_map:
        if abs(score_map[u] - score_map[v]) <= SCORE_SIM_THRESHOLD:
            bonus += 1
    d["weight"] = d.get("weight", 1) + bonus

print("Contoh edge dengan bobot:")
for u, v, d in list(G.edges(data=True))[:5]:
    print(u, "-", v, "-> weight:", d["weight"])

## Tahap 3 — Community Detection (Louvain)

Menjalankan Louvain community detection di atas graph berbobot dari Tahap 2. Karena edge dasarnya
berasal dari kesamaan sub_industri×region, community yang terbentuk seharusnya mendekati pengelompokan
itu sendiri — ini jadi **validasi** bahwa graph well-formed. Insight barunya muncul dari bobot tambahan
(branch & score similarity), yang bisa memecah satu grup sub_industri×region jadi beberapa sub-cluster
kalau profil risikonya sebenarnya berbeda-beda.

In [ ]:
if HAVE_NX_LOUVAIN:
    communities = louvain_communities(G, weight="weight", seed=42)
elif HAVE_PY_LOUVAIN:
    partition = community_louvain.best_partition(G, weight="weight", random_state=42)
    communities_dict = {}
    for node, cid in partition.items():
        communities_dict.setdefault(cid, set()).add(node)
    communities = list(communities_dict.values())
else:
    raise ImportError("Butuh networkx>=3.0 (louvain_communities) atau package 'python-louvain'.")

print(f"Jumlah cluster terbentuk: {len(communities)}")

# mapping nik -> cluster_id
node_to_cluster = {}
for cid, nodes in enumerate(communities):
    for n in nodes:
        node_to_cluster[n] = cid

df["cluster_id"] = df[COL["id"]].map(node_to_cluster)
df["cluster_id"] = df["cluster_id"].fillna(-1).astype(int)  # -1 = nasabah tanpa edge (isolated node)

print(df["cluster_id"].value_counts().head(10))

## Tahap 4 — Metrik Risiko per Cluster

Untuk tiap cluster, hitung:
- **Ukuran cluster** (jumlah nasabah) → cluster besar = concentration risk kalau skornya rendah
- **Rata-rata & std eligibility_score** → std tinggi = profil mirip tapi decision beragam (indikasi inkonsistensi scoring)
- **Proporsi nasabah "Tidak Layak" / low score** dalam cluster → proxy densitas risiko

In [ ]:
LOW_SCORE_THRESHOLD = 0.40  # sejalan dengan tier 'Tidak Layak' yang sudah didiskusikan

agg = {}
if COL["score"] in df.columns:
    agg[COL["score"]] = ["mean", "std", "count"]

cluster_summary = df.groupby("cluster_id").agg(agg)
cluster_summary.columns = ["_".join(c) for c in cluster_summary.columns]
cluster_summary = cluster_summary.rename(columns={
    f"{COL['score']}_mean": "avg_score",
    f"{COL['score']}_std": "std_score",
    f"{COL['score']}_count": "n_nasabah",
})

if COL["score"] in df.columns:
    low_share = (
        df.assign(is_low=df[COL["score"]] < LOW_SCORE_THRESHOLD)
          .groupby("cluster_id")["is_low"].mean()
          .rename("low_score_share")
    )
    cluster_summary = cluster_summary.join(low_share)

# ambil label dominan sub_industri|region per cluster biar gampang dibaca
dominant_attr = (
    df.groupby("cluster_id")["attr_node"]
      .agg(lambda x: x.value_counts().idxmax())
      .rename("dominant_attr")
)
cluster_summary = cluster_summary.join(dominant_attr)

cluster_summary = cluster_summary[cluster_summary.index != -1]  # exclude isolated nodes
cluster_summary = cluster_summary.sort_values("low_score_share", ascending=False)

cluster_summary.head(15)

In [ ]:
print("Top 5 cluster dengan konsentrasi risiko tertinggi (low_score_share):")
display(cluster_summary.head(5))

print("\nTop 5 cluster dengan inkonsistensi scoring tertinggi (std_score, min 10 nasabah):")
display(cluster_summary[cluster_summary["n_nasabah"] >= 10].sort_values("std_score", ascending=False).head(5))

## Tahap 5 — Centrality: Nasabah Representatif per Cluster

Degree centrality dalam graph (dibatasi per cluster) dipakai untuk mencari nasabah yang paling
"tersambung" dalam cluster-nya — berguna sebagai contoh kasus konkret tiap segmen di presentasi.

In [ ]:
representative_rows = []

for cid, nodes in enumerate(communities):
    if len(nodes) < 3:
        continue
    subG = G.subgraph(nodes)
    centrality = nx.degree_centrality(subG)
    if not centrality:
        continue
    top_node = max(centrality, key=centrality.get)
    row = df[df[COL["id"]] == top_node].iloc[0]
    representative_rows.append({
        "cluster_id": cid,
        "representative_nik": top_node,
        "sub_industri": row.get(COL["sub_industri"]),
        "region": row.get(COL["region"]),
        "eligibility_score": row.get(COL["score"], None),
        "centrality": centrality[top_node],
    })

representatives_df = pd.DataFrame(representative_rows).sort_values("cluster_id")
representatives_df.head(15)

## Tahap 6 — Visualisasi di Level Cluster

Karena graph nasabah-individual bisa sangat padat, visualisasi utama dibuat di **level cluster**:
node = cluster, ukuran node = jumlah nasabah, warna = rata-rata eligibility_score (merah = risiko tinggi,
hijau = risiko rendah). Ini jauh lebih terbaca untuk audience non-teknis.

In [ ]:
# Bangun graph antar-cluster: edge kalau ada nasabah di dua cluster berbeda yang terhubung erat di G asli
# (opsional — kalau cluster sudah cukup terpisah, graph ini bisa berupa node-node tanpa banyak edge)
cluster_summary_plot = cluster_summary.reset_index()

fig, ax = plt.subplots(figsize=(10, 8))

CG = nx.Graph()
for _, r in cluster_summary_plot.iterrows():
    CG.add_node(int(r["cluster_id"]), size=r["n_nasabah"], score=r.get("avg_score", 0.5))

pos = nx.spring_layout(CG, seed=42, k=0.8)

sizes = [CG.nodes[n]["size"] * 15 for n in CG.nodes]
scores = [CG.nodes[n]["score"] for n in CG.nodes]

nodes_drawn = nx.draw_networkx_nodes(
    CG, pos, node_size=sizes, node_color=scores, cmap=cm.RdYlGn, vmin=0, vmax=1, ax=ax
)
nx.draw_networkx_labels(
    CG, pos,
    labels={n: cluster_summary_plot.loc[cluster_summary_plot["cluster_id"] == n, "dominant_attr"].values[0]
            for n in CG.nodes},
    font_size=7, ax=ax
)

plt.colorbar(nodes_drawn, ax=ax, label="Avg Eligibility Score")
ax.set_title("Cluster-level View: Ukuran = Jumlah Nasabah, Warna = Avg Eligibility Score")
ax.axis("off")
plt.tight_layout()
plt.show()

## Tahap 7 — Generate Insight / Narasi Otomatis

Auto-generate poin-poin narasi dari cluster dengan risiko tinggi dan cluster dengan inkonsistensi
scoring tinggi, siap ditempel ke slide/laporan.

In [ ]:
insights = []

top_risk = cluster_summary.head(3)
for cid, row in top_risk.iterrows():
    insights.append(
        f"- Cluster '{row['dominant_attr']}' (cluster_id={cid}, n={int(row['n_nasabah'])} nasabah) "
        f"memiliki konsentrasi risiko tertinggi: {row['low_score_share']*100:.1f}% nasabah berada di "
        f"bawah ambang skor {LOW_SCORE_THRESHOLD}, dengan rata-rata eligibility_score {row['avg_score']:.2f}. "
        f"Perlu perhatian khusus dalam kebijakan kredit untuk segmen ini."
    )

inconsistent = cluster_summary[cluster_summary["n_nasabah"] >= 10].sort_values("std_score", ascending=False).head(3)
for cid, row in inconsistent.iterrows():
    insights.append(
        f"- Cluster '{row['dominant_attr']}' (cluster_id={cid}, n={int(row['n_nasabah'])} nasabah) "
        f"menunjukkan variasi eligibility_score tinggi (std={row['std_score']:.2f}) meski profil nasabah "
        f"secara atribut mirip — indikasi perlu review konsistensi kriteria scoring untuk segmen ini."
    )

print("\n".join(insights))

## Penutup & Next Steps

- Notebook ini bisa langsung disambungkan ke pipeline existing capstone (setelah `master_scored.csv` di-generate ulang).
- Kalau ingin memperkuat use case **RM concentration/collusion risk** (bukan hanya industri), diperlukan
  augmentasi kecil ke synthetic data generator: inject sejumlah kecil kasus overlap sengaja
  (shared alamat/agunan/no HP antar nasabah dalam satu RM) sebagai "ground truth" yang bisa dideteksi
  graph — karena data generator saat ini membuat atribut sensitif unik per nasabah, sehingga graph
  nasabah-nasabah untuk kasus itu saat ini tidak akan menghasilkan edge yang meaningful.
- Insight dari Tahap 7 bisa langsung dipakai sebagai bullet point di section baru slide capstone
  ("Graph Analytics: Industry & Region Risk Concentration").